# Train L2 (TemporalCrossEntityAttention) — V3 temporal player/glob pretrain

Pretrains a **TEMPORAL** L2 (`TemporalCrossEntityAttention`) plus the V2-style player/global head menu on the cross_entity dataset. L0 (PlanetEncoder, FleetEncoder) and L1 (PlanetEntityEncoder) are loaded from prior runs and frozen.

**What's temporal about it.** Each history frame gets its own CLS token + 5 owner-summary tokens (`[self, opp1, opp2, opp3, neutral]`), and *all* frames live inside a single L2 attention stack under a **block-causal time mask**: frame `t` attends only to frames `<= t` (and freely within a frame). The supervised player heads still read only the first 4 real-player slots; the neutral slot remains available as context/diagnostics.

**Deep supervision (~T× denser than V2).** The value/outcome head menu is identical to V2 — per-player (`winner`, `leader_k`), pairwise (`winner_pair`, `leader_k_pair`, `current_pair_ahead`), and learner-relative (`is_ahead_k`, `score_adv_k`, `score_adv_end`, `turns_left`) — but here every head is supervised at **every frame** against that frame's own labels, instead of only the single current/anchor frame as in V2. The deployed PPO critic reads the **last frame** `preds[:, -1]`.

**`--head-set v3`.** Passing `--head-set v3` (below) selects `CrossEntityPretrainModelV3` and auto-enables per-frame label stacking in the dataset — `train()` wires `V3_PER_FRAME_LABEL_KEYS` into `CachedCrossEntitySnapshotDataset` for you; the notebook just passes the flag. Because per-frame labels are stacked from history frames, **V3 requires a `.pt` cache** (`--cross-cache-path`); the CSV walk only carries current-frame labels.

**History window:** `HISTORY_OFFSETS = (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)` — uniform 5-turn spacing, 10 slots, ~50-turn lookback. `step_embed` sized to n_steps=10.

**Data path: prebuilt `.pt` cache.** Pulls `cross_entity_cache_{tiny,100k,full}.pt` (`DATASET` selector below; default `'100k'`, the ~21 GB balanced sample) plus the code/weights bundle and `manifest.json`. If a chunk manifest exists (`<cache>.manifest.json` or `<cache-stem>.manifest.json`), the cache chunks are downloaded in parallel and reassembled; otherwise the notebook falls back to the single `.pt` object. `train()` slices the cache into 80/10/10 train/val/test via the manifest's per-split stem lists — no CSV walk.

## Inputs from GCS

```
gs://orbit-wars-shipping/cross_entity/
  code.tgz                       # agents/ + scripts/
  weights.tgz                    # frozen L0 (planet, fleet, comet) d=256
  cross_entity_cache_tiny.pt     # smoke cache (few episodes)
  cross_entity_cache_100k.pt     # ~21 GB balanced sample (default)
  cross_entity_cache_full.pt     # all episodes
  entity_encoder_best.pt         # frozen L1 (May-21 baseline)
  manifest.json                  # split lists
```


## 0. Config

In [ ]:
BATCH_SIZE  = 256
EPOCHS      = 10
LR          = 1e-3
NUM_WORKERS = 2
NUM_LOAD_WORKERS = 8

# V3 reads a prebuilt .pt cache (per-frame supervision needs it). Pick the
# corpus size; v3 is the real run, so default to the 21 GB balanced sample.
#   tiny -> cross_entity_cache_tiny.pt  (end-to-end smoke)
#   100k -> cross_entity_cache_100k.pt  (~21 GB balanced sample, default)
#   full -> cross_entity_cache_full.pt  (all episodes)
DATASET = '100k'
DATASET_CACHE = {
    'tiny': 'cross_entity_cache_tiny.pt',
    '100k': 'cross_entity_cache_100k.pt',
    'full': 'cross_entity_cache_full.pt',
}
CACHE_OBJECT = DATASET_CACHE[DATASET]
print(f'batch={BATCH_SIZE}  epochs={EPOCHS}  lr={LR}  workers={NUM_WORKERS}  load_workers={NUM_LOAD_WORKERS}  dataset={DATASET} ({CACHE_OBJECT})')

## 1. Authenticate + pull bundle

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/cross_entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, shutil, subprocess, time, concurrent.futures, json
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# Wipe stale extracted state before parallel staging starts.
for rel in ('agents', 'scripts', 'ckpts', 'data/datasets', 'data/runs'):
    shutil.rmtree(WORK / rel, ignore_errors=True)
(WORK / 'data/datasets').mkdir(parents=True, exist_ok=True)
for rel in ('cross_entity', 'entity', 'fleet', 'planet'):
    (WORK / 'data/datasets' / rel).mkdir(parents=True, exist_ok=True)

# Cache lands next to the cross_entity CSVs so the manifest sits beside it.
CACHE_PATH = WORK / 'data/datasets/cross_entity' / CACHE_OBJECT

def cp(src, dst, *, force=True):
    dst = Path(dst)
    if dst.exists() and not force:
        return dst.name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'  pulling {src} → {dst.name} ...', flush=True)
    subprocess.run(['gcloud', 'storage', 'cp', src, str(dst)], check=True)
    return dst.name, time.time() - t0, dst.stat().st_size

def pull_cache():
    """Pull CACHE_OBJECT. Prefer chunk manifest for parallel cache download."""
    manifest_local = WORK / f'{CACHE_OBJECT}.manifest.json'
    if manifest_local.exists():
        manifest_local.unlink()
    manifest_candidates = [
        f'{BUCKET}/{CACHE_OBJECT}.manifest.json',
        f'{BUCKET}/{Path(CACHE_OBJECT).stem}.manifest.json',
    ]
    manifest_obj = None
    for cand in manifest_candidates:
        try:
            subprocess.run(
                ['gcloud', 'storage', 'cp', cand, str(manifest_local)],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
            manifest_obj = cand
            break
        except subprocess.CalledProcessError:
            continue
    if manifest_obj is None:
        print(f'  no chunk manifest for {CACHE_OBJECT}; pulling single object', flush=True)
        return cp(f'{BUCKET}/{CACHE_OBJECT}', CACHE_PATH)

    manifest = json.loads(manifest_local.read_text())
    chunks = manifest.get('chunks') or manifest.get('parts') or []
    if not chunks:
        print(f'  empty chunk manifest for {CACHE_OBJECT}; pulling single object', flush=True)
        return cp(f'{BUCKET}/{CACHE_OBJECT}', CACHE_PATH)

    part_dir = WORK / 'cache_parts'
    shutil.rmtree(part_dir, ignore_errors=True)
    part_dir.mkdir(parents=True, exist_ok=True)
    max_part_workers = max(1, min(int(NUM_LOAD_WORKERS), len(chunks)))
    print(f'  pulling {len(chunks)} cache chunks with {max_part_workers} workers ...', flush=True)
    t0 = time.time()

    def pull_part(spec):
        name = spec['name']
        src = name if str(name).startswith('gs://') else f'{BUCKET}/{name}'
        dst = part_dir / Path(name).name
        subprocess.run(['gcloud', 'storage', 'cp', src, str(dst)], check=True)
        expected = int(spec.get('bytes', dst.stat().st_size))
        actual = dst.stat().st_size
        if actual != expected:
            raise RuntimeError(f'chunk size mismatch for {name}: {actual} != {expected}')
        return dst, actual

    parts = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_part_workers) as pool:
        futs = [pool.submit(pull_part, spec) for spec in chunks]
        for fut in concurrent.futures.as_completed(futs):
            parts.append(fut.result())

    if CACHE_PATH.exists():
        CACHE_PATH.unlink()
    with CACHE_PATH.open('wb') as out:
        for spec in chunks:
            part = part_dir / Path(spec['name']).name
            with part.open('rb') as fh:
                shutil.copyfileobj(fh, out, length=64 * 1024 * 1024)
    actual = CACHE_PATH.stat().st_size
    expected_total = int(manifest.get('total_bytes', actual))
    if actual != expected_total:
        raise RuntimeError(f'cache reassembly size mismatch: {actual} != {expected_total}')
    return CACHE_PATH.name, time.time() - t0, actual

def pull_and_stage(src, dst, stage):
    if stage == 'cache':
        name, dt, size = pull_cache()
    else:
        name, dt, size = cp(src, dst)
    t0 = time.time()
    if stage in ('code', 'weights'):
        subprocess.run(['tar', 'xzf', str(dst)], check=True)
    return name, dt, size, stage, time.time() - t0

TASKS = [
    (f'{BUCKET}/code.tgz',                  WORK / 'code.tgz',                 'code'),
    (f'{BUCKET}/weights.tgz',               WORK / 'weights.tgz',              'weights'),
    (f'{BUCKET}/entity_encoder_best.pt',    WORK / 'entity_encoder_best.pt',   'file'),
    (f'{BUCKET}/manifest.json',             WORK / 'manifest.json',            'file'),
    (f'{BUCKET}/{CACHE_OBJECT}',            CACHE_PATH,                        'cache'),
]
T_START = time.time()
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(TASKS)) as pool:
    futs = [pool.submit(pull_and_stage, s, d, stage) for s, d, stage in TASKS]
    for f in concurrent.futures.as_completed(futs):
        results.append(f.result())
for name, dt, size, stage, extract_dt in sorted(results, key=lambda t: -t[2]):
    size_msg = ' streamed' if size < 0 else f'{size / 1024 / 1024:>9.1f} MB'
    print(f'  {name:<34s} {size_msg}  pull={dt:5.1f}s  stage={extract_dt:5.1f}s  {stage}')
# Manifest lives next to the cross_entity cache.
(WORK / 'data/datasets/cross_entity').mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'manifest.json', WORK / 'data/datasets/cross_entity/manifest.json')
print(f'\ncache: {CACHE_PATH}')
print(f'total wall: {time.time()-T_START:.1f}s')

In [ ]:
# Parallel staging happened in the previous cell. This cell only clears
# stale imports and prints a quick filesystem sanity check.
import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
!du -sh data/datasets/*
!ls -la

## 1b. Verify HISTORY_OFFSETS = T=10

In [ ]:
import agents
from agents.transformer_v2.history import HISTORY_OFFSETS, N_HISTORY
print(f'agents module: {agents.__file__}')
print(f'HISTORY_OFFSETS: {HISTORY_OFFSETS}')
assert HISTORY_OFFSETS == (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)
assert N_HISTORY == 10

import torch
# V3 temporal L2 builds with T=10 and 5 slot tokens: 4 players + neutral.
from agents.transformer_v2.aggregator import TemporalCrossEntityAttention
cross = TemporalCrossEntityAttention(d_model=256, n_steps=10, n_players=5)
assert cross.step_embed.shape == (10, 256), cross.step_embed.shape
assert cross.n_players == 5, cross.n_players
print(f'TemporalCrossEntityAttention OK: step_embed {tuple(cross.step_embed.shape)}, slot tokens={cross.n_players}')

# Full V3 pretrain model (temporal trunk + per-frame head menu) builds.
from agents.transformer_v2.pretrain.cross_entity import CrossEntityPretrainModelV3
m = CrossEntityPretrainModelV3(d_model=256)
assert m.cross.step_embed.shape == (10, 256), m.cross.step_embed.shape
assert m.cross.n_players == 5, m.cross.n_players
print(f'CrossEntityPretrainModelV3 OK: step_embed {tuple(m.cross.step_embed.shape)}, slot tokens={m.cross.n_players} (4 players + neutral)')

## 2. Verify GPU

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 3. Stage L0 + L1 ckpts

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
ENTITY_RUN_DIR = Path('/content/orbit-wars/ckpts/entity')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, ENTITY_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/entity_encoder_best.pt', ENTITY_RUN_DIR / 'entity_encoder_best.pt')

import torch
for tag, p in (('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
                ('fleet',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt'),
                ('entity', ENTITY_RUN_DIR / 'entity_encoder_best.pt')):
    c = torch.load(p, map_location='cpu', weights_only=False)
    print(f'{tag:6s} ckpt: d_model={c["config"]["d_model"]}, epoch={c["epoch"]}, '
          f'use_traj_branch={c["config"].get("use_traj_branch")}')
    assert c['config']['d_model'] == 256

## 4. Train L2 (V3 temporal, deep supervision)

`--head-set v3` selects `CrossEntityPretrainModelV3` (per-frame CLS + 5 owner-slot tokens under a block-causal time mask: 4 real players + neutral) and supervises the value/outcome heads at every frame. `--cross-cache-path` points at the prebuilt cache; `train()` slices it into the 80/10/10 splits via the manifest and auto-stacks per-frame labels (`V3_PER_FRAME_LABEL_KEYS`) for the deep-supervision loss.

In [ ]:
D_MODEL    = 256
WEIGHT_DECAY = 1e-4
SEED       = 1729
DEVICE     = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'temporal_v3_playerglob_{DATASET}_d{D_MODEL}_b{BATCH_SIZE}_{EPOCHS}ep_lr{LR:g}_{TS}'
OUT_DIR = f'data/runs/cross_entity/{RUN_TAG}'
print('out dir:', OUT_DIR)

In [ ]:
!python -u -m agents.transformer_v2.pretrain.cross_entity \
  --train-mode frozen \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --planet-run-dir $PLANET_RUN_DIR \
  --entity-run-dir $ENTITY_RUN_DIR \
  --out-dir $OUT_DIR \
  --cross-cache-path $CACHE_PATH \
  --d-model $D_MODEL \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --head-set v3 \
  --seed $SEED \
  --num-load-workers $NUM_LOAD_WORKERS \
  --num-workers $NUM_WORKERS \
  --device $DEVICE

## 5. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
dst_parent = f'{BUCKET}/runs/'
subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent], check=True)
print(f'uploaded to: {dst_parent}{src.name}/')
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{dst_parent}{src.name}/'], check=False)